In [0]:
#Streaming data for preprocessing

display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/telegram_project"))

In [0]:
# create folder raw
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/telegram_project/raw")

# create folder processed
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/telegram_project/processed")

In [0]:
from huggingface_hub import login 
from dotenv import load_dotenv
import os 
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
login(HF_TOKEN)



In [0]:
pip install datasets 

In [0]:
from datasets import load_dataset 
ds = load_dataset(
    "leonardoblas/us_election_2024_telegram_distilled",
    split = "train",
    streaming = True 
)

for i,row in enumerate(ds):
    print(row)
    if i >= 4:
        break 

In [0]:
row = next(iter(ds))
print(row)
print()
print(row.keys())

In [0]:
def pick(row,candidates,default = None):
    for c in candidates:
        if c in row and row[c] is not None:
            return row[c]
    return default

import pandas as pd 
import os
output_dir = "/Volumes/workspace/default/telegram_project/processed"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
output_path = os.path.join(output_dir, "messages_clean.parquet")

max_rows = 500000
batch_size = 5000

buffer = []
written = 0
first_write = True

for i,row in enumerate(ds):
    text = pick(row, ["text", "message", "content", "body"], "")
    chat_id = pick(row, ["chat_id", "peer_id", "channel_id"])
    channel = pick(row, ["channel", "channel_name", "chat_name", "peer_name"])
    date = pick(row, ["date", "timestamp", "created_at"])
    views = pick(row, ["views", "view_count"], 0)
    forwards = pick(row, ["forwards", "forward_count"], 0)
    forward_from_chat = pick(row, ["forward_from_chat", "fwd_from_chat", "forward_source_chat"])
    message_id = pick(row, ["message_id", "id"])

    if text is None:
        text = ""
    text = str(text).strip()
    if views is None:
        views = 0
    if forwards is None:
        forwards = 0
    try:
        views = int(views)
    except:
        views = 0

    try:
        forwards = int(forwards)
    except:
        forwards = 0
    has_text = len(text) > 0
    has_signal = (forwards > 0) or (views > 0) or (forward_from_chat is not None)
    if has_text and has_signal:
        buffer.append({
            "message_id": message_id,
            "channel": channel,
            "chat_id": chat_id,
            "date": date,
            "text": text,
            "views": views,
            "forwards": forwards,
            "forward_from_chat": forward_from_chat
        })
    if len(buffer) >= batch_size:
        batch_df = pd.DataFrame(buffer)

        if first_write:
            batch_df.to_parquet(output_path, index=False)
            first_write = False
        else:
            old_df = pd.read_parquet(output_path)
            merged_df = pd.concat([old_df, batch_df], ignore_index=True)
            merged_df.to_parquet(output_path, index=False)

        written += len(buffer)
        print(f"Written so far: {written}")
        buffer = []

        if written >= max_rows:
            break
if buffer and written < max_rows:
    batch_df = pd.DataFrame(buffer)

    if first_write:
        batch_df.to_parquet(output_path, index=False)
    else:
        old_df = pd.read_parquet(output_path)
        merged_df = pd.concat([old_df, batch_df], ignore_index=True)
        merged_df.to_parquet(output_path, index=False)

    written += len(buffer)

print(f"Done. Total rows written: {written}")
print(f"Saved to: {output_path}")


In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/telegram_project/processed/"))